# CS604 Colab launcher — Qian

Runtime: **A100 40GB GPU**, High-RAM off. Images stay on Colab disk (`/content/data`). Checkpoints and logs go to the shared Drive folder **CS604_RUNS**.

The GitHub repo is private, so the clone cell will ask for a Personal Access Token (hidden). Do not paste the token into the notebook text or commit it.

**Drive:** everyone must have `CS604_RUNS` under *My Drive* (create it there, or open the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR) → Organize → **Add shortcut to Drive** → My Drive, name it `CS604_RUNS`). A Drive *link* is not a file path Colab can write to.

Change `METHOD`, `TEAM`, and `BRANCH` below before you run.

In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
# Must match a folder (or shortcut) in My Drive — not the browser URL.
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "qq_branch_v2"
METHOD = "lastblock_attention"  # linear_probe | full_ft | lora_r8 | bitfit | last_blocks | prototype_probe | prototype_lora_r8
TEAM = "08"
PROTO_PT = f"{DRIVE_OUT}/prototypes.pt"

## 1. Clone or pull the repo

In [ ]:
import os
from getpass import getpass

token = getpass("GitHub PAT (hidden; repo scope). Leave empty if the repo is public: ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    if token:
        !git -C {REPO_DIR} pull {clone_url} {BRANCH}
    else:
        !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {clone_url} {REPO_DIR}

%cd {REPO_DIR}
!git branch --show-current
!git rev-parse --short HEAD

## 2. Install requirements

In [ ]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])

## 4. Download data to `/content/data`

In [ ]:
!python scripts/download_data.py --dest {DATA_DIR}

## 4b. Build class prototypes (once per backbone)

Van's ~43.9% zero-shot head. Run **once**, then reuse `PROTO_PT` for `prototype_probe` / `prototype_lora_r8`. Takes a while (embeds train images). Skip if the file already exists on Drive.

In [ ]:
from pathlib import Path

if Path(PROTO_PT).is_file():
    print("Already have", PROTO_PT)
else:
    !python scripts/build_prototypes.py --config configs/prototype_probe.yaml --data_root {DATA_DIR} --output {PROTO_PT}

## Review settings (table)

Loads `configs/{METHOD}.yaml` (merged with `base.yaml`) so you can check every parameter before spending a GPU hour.

In [ ]:
import pandas as pd

from src.utils import apply_overrides, load_config

methods = ["lastblock_attention", "full_ft", "linear_probe", "lora_r8", "bitfit", "last_blocks", "prototype_probe", "prototype_lora_r8", "adapter", "adaptformer"]
merged = {name: load_config(f"configs/{name}.yaml").to_dict() for name in methods}
keys = []
for d in merged.values():
    for k in d:
        if k not in keys:
            keys.append(k)

compare = pd.DataFrame(
    {name: [merged[name].get(k, "") for k in keys] for name in methods},
    index=keys,
)
compare.index.name = "parameter"
print("Resolved parameters per method (base.yaml already merged in):")
display(compare)

chosen = load_config(f"configs/{METHOD}.yaml")
apply_overrides(chosen, data_root=DATA_DIR, output_dir=DRIVE_OUT)
resolved = pd.DataFrame(list(chosen.to_dict().items()), columns=["parameter", "value"])
print(f"\nThis run (METHOD={METHOD!r}, plus Colab data_root / output_dir):")
display(resolved)

## 5. Train (checkpoints on Drive)
`lastblock_attention` uses the linear-probe validation recipe with prototype initialization but trains only final-block attention and the classifier. It runs for 144 training minutes with no epoch ceiling and 5% time-based warmup; cosine decay follows elapsed training time. Setup, validation and saving are excluded from the timer.


In [ ]:
import torch
from src.utils import load_config, apply_overrides

assert torch.cuda.is_available(), \
    "Select a GPU runtime; CPU training is too slow for this budget."

check = load_config(f"configs/{METHOD}.yaml")
apply_overrides(check, data_root=DATA_DIR, output_dir=DRIVE_OUT)

print("GPU:", torch.cuda.get_device_name(0))
print("Resolved config:", check.to_dict())

!git rev-parse --short HEAD

if METHOD == "lastblock_attention":
    assert check.epochs is None and check.max_train_minutes == 144

!python -u -m src.train --config configs/{METHOD}.yaml --data_root {DATA_DIR} --output_dir {DRIVE_OUT}


## 6. Predict

Set `RUN_NAME` to the folder printed by training. Select `last.pt`, or `best.pt` if labeled validation generated it. Predictions are saved inside this run folder. Test prediction runs only when test data is present.


In [ ]:
from pathlib import Path
import subprocess

RUN_NAME = "REPLACE_WITH_RUN_DIR"
RUN_DIR = Path(DRIVE_OUT) / RUN_NAME
CKPT = str(RUN_DIR / "last.pt")
assert Path(CKPT).is_file(), f"Checkpoint missing: {CKPT}"

# Use the saved settings from this exact run.
def predict(split):
    subprocess.run([
        "python", "-m", "src.predict",
        "--config", str(RUN_DIR / "config.yaml"),
        "--ckpt", CKPT, "--split", split, "--team", TEAM,
        "--data_root", DATA_DIR, "--out_dir", str(RUN_DIR),
    ], check=True)

predict("val")
__import__("shutil").make_archive(str(RUN_DIR / f"val_team_{TEAM}"), "zip", root_dir=str(RUN_DIR), base_dir=f"val_team_{TEAM}.txt")
if (Path(DATA_DIR) / "test.txt").is_file() and (Path(DATA_DIR) / "test").is_dir():
    predict("test")
else:
    print("Test data not available; skipping test prediction.")


## 7. Loss curves

Plot the losses recorded during training, after training finishes. Save and display the plots in this experiment’s Drive folder. Validation curves require labeled validation.


In [ ]:
from IPython.display import display, Image

FIGURE_DIR = RUN_DIR / "figures"
subprocess.run([
    "python", "scripts/plot_curves.py",
    "--run_dir", str(RUN_DIR), "--out_dir", str(FIGURE_DIR),
], check=True)

for plot in sorted(FIGURE_DIR.glob("*.png")):
    print(plot.name)
    display(Image(filename=str(plot)))


## 8. Inference latency

Load the trained checkpoint selected in section 6. Measure forward-pass latency on synthetic images at batch sizes 1 and 64, excluding preprocessing and transfers. Use 30 warm-ups and 200 timed iterations. Save results to `inference_latency.json`. This runs outside the training timer; keep GPU and measurement settings consistent between experiments.


In [ ]:
import json
import torch

from src.utils import load_config, apply_overrides, set_seed
from src.model import build_model
from src.predict import load_checkpoint
from scripts.benchmark_latency import measure

assert torch.cuda.is_available(), "A GPU runtime is required."
cfg = load_config(RUN_DIR / "config.yaml")
# Partial PEFT checkpoints require the original pretrained backbone.
checkpoint_info = torch.load(CKPT, map_location="cpu", weights_only=False)
partial = bool(checkpoint_info.get("trainable_only", False))
del checkpoint_info
apply_overrides(cfg, pretrained=partial)
set_seed(int(cfg.seed))

device = torch.device("cuda")
model = build_model(cfg).to(device)
load_checkpoint(model, CKPT, device)
model.eval()

warmup = 30
iterations = 200
use_bf16 = bool(cfg.bf16)
results = {
    "checkpoint": CKPT,
    "gpu": torch.cuda.get_device_name(0),
    "gpu_capacity_GiB": torch.cuda.get_device_properties(0).total_memory / 1024**3,
    "torch_version": str(torch.__version__),
    "precision": "bf16" if use_bf16 else "fp32",
    "img_size": int(cfg.img_size),
    "warmup_iterations": warmup,
    "timed_iterations": iterations,
    "scope": "Forward pass on synthetic inputs; excludes loading, preprocessing and transfers",
    "measurements": [],
}

try:
    for batch_size in (1, 64):
        batch = torch.randn(
            batch_size, 3, int(cfg.img_size), int(cfg.img_size), device=device,
        )
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_bf16):
            ms_per_image, _ = measure(model, batch, device, warmup, iterations)
        results["measurements"].append({
            "batch_size": batch_size,
            "ms_per_image": ms_per_image,
            "ms_per_batch": ms_per_image * batch_size,
        })
        del batch
finally:
    del model
    torch.cuda.empty_cache()

output = RUN_DIR / "inference_latency.json"
output.write_text(json.dumps(results, indent=2), encoding="utf-8")
print(json.dumps(results, indent=2))
print("Saved:", output)
